In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

_install('torch')
_install('matplotlib')
_install('numpy')


In [ ]:
import math, os
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


# MoE 代码实战：学习实现 vs 工程实现

基于论文 *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer*（Shazeer et al., 2017）*，
用 **二维合成分类任务** 演示 Mixture of Experts（MoE，混合专家模型） 的核心思想。

本 Notebook 包含两条并行路径，使用 **相同的任务和数据**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解门控、专家、路由、辅助损失 | 用更短代码快速搭建可训练的 MoE |
| 实现方式 | L1：从零手写 `NoisyTop1Gate` + `SparseMoELayer` | E3：原生 PyTorch 下的紧凑式 `CompactMoEClassifier` |
| 代码量 | ~140 行核心代码 | ~55 行核心代码 |
| 适合场景 | 面试准备、原理理解、分析路由行为 | 快速实验、作业实现、原型验证 |

> 两条路径不是两套无关代码，而是同一套 MoE 思想在不同抽象层级上的表达。


## i. 论文与任务背景

### 论文背景

MoE 的核心思想是：

- 准备多个 **专家网络**（experts）
- 用一个 **门控网络**（gate）为每个输入选择少数专家
- 只让少数专家参与计算，从而在较低计算成本下提升参数容量

在现代大模型中，MoE 常用于替换 Transformer 中的 FFN 层。Switch Transformer 进一步将路由简化为 **top-1 路由**，Mixtral 则常见 **top-2 路由**。

### 为什么这里选二维合成分类任务

MoE 本质上是一种 **条件计算层**，而不是必须绑定某个具体任务。为了让 Notebook 可以在 Colab 免费环境和 CPU 上稳定运行，这里选用一个二维输入的分段分类任务：

- 数据量小，训练快
- 决策边界是分段的，适合专家分工
- 可以直接可视化专家路由行为

### 本 Notebook 不覆盖什么

- 不复现大语言模型中的分布式 all-to-all 通信
- 不实现专家容量裁剪（expert capacity）
- 不实现 token 级 Transformer MoE 替换，只聚焦最小可运行概念验证


## ii. 最小必要理论

这一节只保留理解后续代码所必需的公式。完整背景请配合 `MoE.md` 阅读。

### 1. 门控网络

给定输入表示 $x$，门控网络输出每个专家的分数：

$$G(x) = \mathrm{Softmax}(xW_g)$$

如果采用稀疏路由，则只保留 top-$k$ 个专家：

$$G(x) = \mathrm{Softmax}(\mathrm{TopK}(xW_g, k))$$

### 2. 专家聚合

第 $i$ 个专家记为 $E_i(x)$，则输出为：

$$y = \sum_{i=1}^{N} G_i(x) \cdot E_i(x)$$

### 3. 负载均衡辅助损失

为防止门控长期偏向少数专家，加入辅助损失：

$$L_{aux} = \alpha \sum_{i=1}^{N} f_i \cdot P_i$$

其中 $f_i$ 表示专家实际负载比例，$P_i$ 表示平均门控概率。

### 4. 训练 vs 推理

训练时常加入 **Noisy Gating** 促进探索；推理时通常关闭噪声，使用确定性路由。


### 组件映射表

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Gating Network | `NoisyTop1Gate` 手写实现 | `nn.Linear` + `F.softmax` | Runnable |
| Top-k / Top-1 Sparse Routing | 显式 `argmax` + one-hot mask | `torch.topk` / `scatter_` | Runnable |
| Expert Networks | `ExpertMLP` 手写实现 | `nn.Sequential` / `nn.ModuleList` | Runnable |
| Weighted Aggregation | 显式加权求和 | `torch.stack` + broadcast / `einsum` | Runnable |
| Auxiliary Load-Balance Loss | 显式计算 `load * importance` | 原生张量运算 | Runnable |
| Noisy Gating | `self.training` 时注入噪声 | `torch.randn_like` + `softplus` | Runnable |
| Expert Capacity | 仅概念解释，不在最小示例中实现 | 常见于 Switch Transformer | Explain-only |
| Top-2 Routing / Mixtral 风格 | 仅解释，不在本示例中实现 | `torch.topk(..., k=2)` | Explain-only |


## 1. 数据准备

我们构造一个二维分段分类任务。输入为 $(x_1, x_2)$，标签规则按 $x_1$ 所在区域变化：

- 左区域：线性边界
- 中区域：圆形边界
- 右区域：正弦边界

这类任务天然适合 MoE：不同专家可以分别擅长不同局部规则。


In [ ]:
def make_piecewise_dataset(n_samples=8000, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2.0, 2.0, size=(n_samples, 2)).astype(np.float32)

    left = x[:, 0] < -0.5
    middle = (x[:, 0] >= -0.5) & (x[:, 0] < 0.5)
    right = x[:, 0] >= 0.5

    y = np.zeros(n_samples, dtype=np.int64)
    y[left] = (x[left, 1] > (0.8 * x[left, 0] + 0.2)).astype(np.int64)
    y[middle] = ((x[middle, 0] ** 2 + x[middle, 1] ** 2) < 0.9).astype(np.int64)
    y[right] = (x[right, 1] > 0.6 * np.sin(3.0 * x[right, 0])).astype(np.int64)

    noise_mask = rng.random(n_samples) < 0.05
    y[noise_mask] = 1 - y[noise_mask]
    return x, y

x_train, y_train = make_piecewise_dataset(6000, seed=42)
x_test, y_test = make_piecewise_dataset(1500, seed=123)

train_ds = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))

print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)


In [ ]:
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(x_train[:, 0], x_train[:, 1], c=y_train, s=6, cmap='coolwarm', alpha=0.5)
ax[0].set_title('Training Data')
ax[0].set_xlabel('x1')
ax[0].set_ylabel('x2')

region_color = np.where(x_train[:, 0] < -0.5, 0, np.where(x_train[:, 0] < 0.5, 1, 2))
ax[1].scatter(x_train[:, 0], x_train[:, 1], c=region_color, s=6, cmap='viridis', alpha=0.5)
ax[1].set_title('Underlying Regions')
ax[1].set_xlabel('x1')
ax[1].set_ylabel('x2')
plt.tight_layout()
plt.show()

sample_x, sample_y = next(iter(train_loader))
print('Batch x shape:', sample_x.shape)
print('Batch y shape:', sample_y.shape)


## 2. 共享组件

两条路径共用同一套超参数、训练循环和评估逻辑。


In [ ]:
# ── 超参数（两条路径共用，集中管理） ──
INPUT_DIM    = 2
HIDDEN_DIM   = 32
NUM_EXPERTS  = 3
NUM_CLASSES  = 2
DROPOUT      = 0.1
LR           = 3e-3
NUM_EPOCHS   = 25
AUX_ALPHA    = 0.05

print({
    'INPUT_DIM': INPUT_DIM,
    'HIDDEN_DIM': HIDDEN_DIM,
    'NUM_EXPERTS': NUM_EXPERTS,
    'NUM_EPOCHS': NUM_EPOCHS,
    'LR': LR,
    'AUX_ALPHA': AUX_ALPHA,
})

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total_count = 0.0, 0, 0
    routing_hist = torch.zeros(NUM_EXPERTS, dtype=torch.long)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, info = model(x)
            loss = criterion(logits, y) + AUX_ALPHA * info['aux_loss']

            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_count += y.size(0)

            top_idx = info['top_idx'].detach().cpu()
            routing_hist += torch.bincount(top_idx, minlength=NUM_EXPERTS)

    return {
        'loss': total_loss / total_count,
        'acc': total_correct / total_count,
        'routing_hist': routing_hist,
    }


def train_model(model, train_loader, test_loader, num_epochs, lr, device):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
    }

    for epoch in range(num_epochs):
        model.train()
        total_loss, total_correct, total_count = 0.0, 0, 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            logits, info = model(x)
            ce_loss = criterion(logits, y)
            loss = ce_loss + AUX_ALPHA * info['aux_loss']

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_count += y.size(0)

        train_loss = total_loss / total_count
        train_acc = total_correct / total_count
        test_metrics = evaluate_model(model, test_loader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_metrics['loss'])
        history['test_acc'].append(test_metrics['acc'])

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{num_epochs}]  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  test_acc={test_metrics['acc']:.4f}")

    return history


## iii. 学习路径：从零实现 MoE

### 组件 1：专家网络（Expert MLP）

每个专家都是一个小型前馈网络，输入输出张量形状相同，便于后续加权聚合。

- 输入：`(batch, hidden_dim)`
- 输出：`(batch, hidden_dim)`

### 组件 2：Noisy Top-1 Gate

这里采用与 Switch Transformer 思路一致的 **top-1 路由**：

1. 先计算所有专家分数
2. 训练时加噪声促进探索
3. 选择分数最高的那个专家
4. 用 one-hot mask 保留该专家的概率

对应公式可以写成：

$$\text{logits} = xW_g + \epsilon \cdot \mathrm{Softplus}(xW_{noise})$$
$$\text{top1}(x) = \arg\max_i \; \text{logits}_i$$

虽然这里为了教学简化为对整个 batch 计算所有专家输出再加权，但路由逻辑与真实 MoE 一致。


In [ ]:
class ExpertMLP(nn.Module):
    """单个专家网络.
    (batch, hidden_dim) -> (batch, hidden_dim)
    """
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class NoisyTop1Gate(nn.Module):
    """Noisy Top-1 门控.
    输入:  (batch, hidden_dim)
    输出:  route_weights (batch, num_experts)
          probs         (batch, num_experts)
          top_idx       (batch,)
          aux_loss      scalar
    """
    def __init__(self, hidden_dim, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.gate = nn.Linear(hidden_dim, num_experts)
        self.noise_proj = nn.Linear(hidden_dim, num_experts)

    def forward(self, x):
        logits = self.gate(x)  # (batch, num_experts)

        if self.training:
            noise_std = F.softplus(self.noise_proj(x)) + 1e-2  # (batch, num_experts)
            logits = logits + torch.randn_like(logits) * noise_std

        probs = F.softmax(logits, dim=-1)  # (batch, num_experts)
        top_idx = torch.argmax(logits, dim=-1)  # (batch,)
        top_mask = F.one_hot(top_idx, num_classes=self.num_experts).float()  # (batch, num_experts)

        route_weights = probs * top_mask

        load = top_mask.mean(dim=0)       # (num_experts,)
        importance = probs.mean(dim=0)    # (num_experts,)
        aux_loss = self.num_experts * torch.sum(load * importance)
        return route_weights, probs, top_idx, aux_loss


_gate = NoisyTop1Gate(HIDDEN_DIM, NUM_EXPERTS)
_h = torch.randn(8, HIDDEN_DIM)
_route_weights, _probs, _top_idx, _aux = _gate(_h)
print('route_weights:', _route_weights.shape)
print('probs        :', _probs.shape)
print('top_idx      :', _top_idx.shape)
print('aux_loss     :', float(_aux))

### 训练 vs 推理的区别

这是 MoE 中非常容易被忽略，但面试很常问的一点。

| 阶段 | 门控行为 | 目的 |
|---|---|---|
| 训练 | `self.training=True`，对 gate logits 注入噪声 | 鼓励探索更多专家，缓解路由塌缩 |
| 推理 | `model.eval()`，关闭噪声，只保留确定性 top-1 路由 | 输出更稳定，可重复 |

这与理论里的 Noisy Gating 完全对应。

### 组件 3：Sparse MoE Layer

输出聚合公式：

$$y = \sum_{i=1}^{N} G_i(x)E_i(x)$$

代码里对应为：

- 先让每个专家产生一个候选输出
- 再按 `route_weights` 做加权和
- 这里采用的是 **概率加权的 top-1 路由**：只保留被选中的专家，但保留该专家对应的 softmax 权重，而不是把它强行归一成 1

张量形状：

- `expert_outputs`: `(batch, num_experts, hidden_dim)`
- `route_weights`: `(batch, num_experts)`
- `mixed`: `(batch, hidden_dim)`

In [ ]:
class SparseMoELayer(nn.Module):
    """显式 MoE 层.
    输入:  (batch, hidden_dim)
    输出:  (batch, hidden_dim)
    """
    def __init__(self, hidden_dim, num_experts, dropout=0.1):
        super().__init__()
        self.gate = NoisyTop1Gate(hidden_dim, num_experts)
        self.experts = nn.ModuleList([
            ExpertMLP(hidden_dim, dropout=dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        route_weights, probs, top_idx, aux_loss = self.gate(x)

        expert_outputs = torch.stack([
            expert(x) for expert in self.experts
        ], dim=1)  # (batch, num_experts, hidden_dim)

        mixed = torch.sum(expert_outputs * route_weights.unsqueeze(-1), dim=1)
        info = {
            'probs': probs,
            'top_idx': top_idx,
            'aux_loss': aux_loss,
        }
        return mixed, info


class LearningMoEClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, num_experts=3, num_classes=2, dropout=0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.moe = SparseMoELayer(hidden_dim, num_experts, dropout=dropout)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.backbone(x)        # (batch, hidden_dim)
        mixed, info = self.moe(h)  # (batch, hidden_dim)
        logits = self.head(mixed)  # (batch, num_classes)
        return logits, info


learning_model = LearningMoEClassifier(
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
    num_experts=NUM_EXPERTS, num_classes=NUM_CLASSES,
    dropout=DROPOUT,
)

_x = torch.randn(16, INPUT_DIM)
_logits, _info = learning_model(_x)
print('logits shape:', _logits.shape)
print('top_idx shape:', _info['top_idx'].shape)
print('aux_loss:', float(_info['aux_loss']))


### 学习路径训练

现在训练手写版 MoE。这里的目标不是追求 SOTA，而是观察：

- 模型能否学会分段规则
- 专家是否出现分工
- 辅助损失是否能让路由更均衡


In [ ]:
print('=== Learning Path: train from scratch ===')
learning_history = train_model(learning_model, train_loader, test_loader, NUM_EPOCHS, LR, device)


In [ ]:
learning_metrics = evaluate_model(learning_model.to(device), test_loader, device)
print('Learning test_acc:', round(learning_metrics['acc'], 4))
print('Learning routing hist:', learning_metrics['routing_hist'].tolist())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(learning_history['train_loss'], label='train')
axes[0].plot(learning_history['test_loss'], label='test')
axes[0].set_title('Learning Path Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(learning_history['train_acc'], label='train')
axes[1].plot(learning_history['test_acc'], label='test')
axes[1].set_title('Learning Path Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].bar([f'Expert {i}' for i in range(NUM_EXPERTS)], learning_metrics['routing_hist'].numpy())
axes[2].set_title('Learning Path Routing Usage')
axes[2].set_xlabel('Expert')
axes[2].set_ylabel('Assigned Samples')
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## iv. 工程路径：更紧凑的原生 PyTorch 实现

由于 MoE 没有像 CNN / Transformer 那样统一成熟的高层工业 API，这里选择 **E3：No industrial lib** 的工程路径。

思路是：

- 仍然使用 `nn.ModuleList` 保存多个专家
- 但把门控、专家调用、加权聚合压缩进一个紧凑类里
- 追求更少样板代码，而不是逐组件展开讲解

### 学习路径 ↔ 工程路径对应关系

| 学习路径（手写） | 工程路径（紧凑写法） | 说明 |
|---|---|---|
| `NoisyTop1Gate` | `self.gate` + `self.noise_proj` | 同一门控思想，更短写法 |
| `ExpertMLP` | `nn.Sequential(...)` experts | 同一专家结构，更少解释层 |
| `SparseMoELayer` | `forward()` 内联完成 | 聚合逻辑被压缩 |
| 显式 `route_weights` 广播乘法 | `torch.einsum` | 数学完全等价 |


In [ ]:
class CompactMoEClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, num_experts=3, num_classes=2, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.stem = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.gate = nn.Linear(hidden_dim, num_experts)
        self.noise_proj = nn.Linear(hidden_dim, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
            ) for _ in range(num_experts)
        ])
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.stem(x)  # (batch, hidden_dim)
        logits_gate = self.gate(h)
        if self.training:
            logits_gate = logits_gate + torch.randn_like(logits_gate) * (F.softplus(self.noise_proj(h)) + 1e-2)

        probs = F.softmax(logits_gate, dim=-1)  # (batch, num_experts)
        top_idx = torch.argmax(logits_gate, dim=-1)
        sparse_mask = F.one_hot(top_idx, num_classes=self.num_experts).float()
        sparse_probs = probs * sparse_mask

        expert_out = torch.stack([expert(h) for expert in self.experts], dim=1)  # (batch, num_experts, hidden_dim)
        mixed = torch.einsum('be,beh->bh', sparse_probs, expert_out)

        load = sparse_mask.mean(dim=0)
        importance = probs.mean(dim=0)
        aux_loss = self.num_experts * torch.sum(load * importance)

        return self.head(mixed), {'top_idx': top_idx, 'probs': probs, 'aux_loss': aux_loss}


torch.manual_seed(42)
np.random.seed(42)
engineering_model = CompactMoEClassifier(
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
    num_experts=NUM_EXPERTS, num_classes=NUM_CLASSES,
    dropout=DROPOUT,
)

_logits, _info = engineering_model(torch.randn(16, INPUT_DIM))
print('engineering logits shape:', _logits.shape)
print('engineering aux_loss:', float(_info['aux_loss']))

In [ ]:
print('=== Engineering Path: compact native PyTorch MoE ===')
engineering_history = train_model(engineering_model, train_loader, test_loader, NUM_EPOCHS, LR, device)


## v. 学习路径 vs 工程路径对比

这一节必须把两条路径放在同一个坐标系里比较，而不是只看谁的准确率更高。


In [ ]:
engineering_metrics = evaluate_model(engineering_model.to(device), test_loader, device)
print('Engineering test_acc:', round(engineering_metrics['acc'], 4))
print('Engineering routing hist:', engineering_metrics['routing_hist'].tolist())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(learning_history['test_acc'], label='learning')
axes[0].plot(engineering_history['test_acc'], label='engineering')
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(learning_history['test_loss'], label='learning')
axes[1].plot(engineering_history['test_loss'], label='engineering')
axes[1].set_title('Test Loss Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

bar_x = np.arange(NUM_EXPERTS)
width = 0.35
axes[2].bar(bar_x - width / 2, learning_metrics['routing_hist'].numpy(), width=width, label='learning')
axes[2].bar(bar_x + width / 2, engineering_metrics['routing_hist'].numpy(), width=width, label='engineering')
axes[2].set_title('Routing Usage Comparison')
axes[2].set_xlabel('Expert')
axes[2].set_ylabel('Assigned Samples')
axes[2].set_xticks(bar_x, [f'Expert {i}' for i in range(NUM_EXPERTS)])
axes[2].legend()
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print({
    'learning_test_acc': round(learning_metrics['acc'], 4),
    'engineering_test_acc': round(engineering_metrics['acc'], 4),
})


| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 把门控、专家、路由、辅助损失逐个拆开，最适合理解内部机制 | 更适合理解如何把 MoE 快速落成一个可训练模型 |
| 代码量 | 较多，显式拆分多个类 | 更少，逻辑集中在一个类中 |
| 灵活性 | 可以逐组件替换、插桩、可视化 | 适合快速改超参数，不适合精细教学 |
| 稳定性 | 更容易因实现细节出错 | 结构更紧凑，工程上更直接 |
| 工业适配度 | 适合研究原型与教学 | 更接近实际实验代码风格 |
| 适用场景 | 面试准备、论文复现、概念验证 | 作业实现、原型搭建、快速试错 |


## vi. 训练 vs 推理差异

虽然本任务是分类，不像生成模型那样存在 teacher forcing / autoregressive 区别，但 **Noisy Gating** 仍然让训练和推理有明确差异。

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `NoisyTop1Gate` 注入噪声，再做 top-1 路由 | `forward()` 中对 gate logits 加噪声 |
| 推理 | `model.eval()` 关闭噪声，使用确定性路由 | `model.eval()` 关闭噪声，使用确定性路由 |
| 核心 insight | 同一门控公式，不同阶段使用不同随机性 | 同一算法，只是抽象层更高 |


In [ ]:
def inspect_points(model, points, device):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(points, dtype=torch.float32).to(device)
        logits, info = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        return {
            'pred': pred,
            'top_idx': info['top_idx'].cpu().numpy(),
            'probs': info['probs'].cpu().numpy(),
        }

demo_points = [
    [-1.5, -0.8],
    [-1.2,  0.9],
    [ 0.0,  0.0],
    [ 0.2,  1.2],
    [ 1.2, -0.7],
    [ 1.4,  0.8],
]

learning_demo = inspect_points(learning_model.to(device), demo_points, device)
engineering_demo = inspect_points(engineering_model.to(device), demo_points, device)

print('=== Learning Path Inference ===')
for p, pred, idx in zip(demo_points, learning_demo['pred'], learning_demo['top_idx']):
    print(f'point={p}  pred={pred}  expert={idx}')

print('\n=== Engineering Path Inference ===')
for p, pred, idx in zip(demo_points, engineering_demo['pred'], engineering_demo['top_idx']):
    print(f'point={p}  pred={pred}  expert={idx}')


## vii. 结果与边界

### 你从学习路径得到什么

- 能直接回答 gate、expert、auxiliary loss 分别做什么
- 能解释为什么要负载均衡、为什么训练时要加噪声
- 能通过路由统计看到专家是否塌缩

### 你从工程路径得到什么

- 知道没有成熟高层库时，如何用原生 PyTorch 快速搭建紧凑版 MoE
- 能在同样任务上快速迭代超参数和结构

### 两条路径的共同边界

这个最小示例没有覆盖真实大模型中的：

- token 级别路由
- 分布式 all-to-all 通信
- expert capacity / token dropping
- top-2 / top-k 更复杂路由

因此，它更适合建立 **MoE 的第一性理解**，而不是直接替代工业级 LLM MoE 训练框架。


## viii. 面试与项目表达

### 高频面试题

**Q1：MoE 为什么能在不显著增加计算量的前提下提升模型容量？**

- 因为每次输入只激活少数专家
- 总参数量可以很大，但每次前向只用其中一小部分
- 所以参数规模和单次计算量被部分解耦

**Q2：MoE 的门控网络到底在学什么？**

- 它在学一个路由函数
- 本质上是把不同输入分配给更合适的专家
- 如果数据具有分段结构，不同专家会自然形成分工

**Q3：为什么会发生 expert collapse？**

- 因为门控更容易持续偏向少数高分专家
- 一旦某些专家更常被选中，它们更新更多，优势会进一步放大
- 这会形成马太效应，导致多数专家几乎不工作

**Q4：辅助损失在解决什么问题？**

- 它约束路由的负载分布更均衡
- 避免所有 token 都涌向少数专家
- 本质是把“效果好”与“负载均衡”同时纳入优化目标

**Q5：Noisy Gating 为什么重要？**

- 没有噪声时，门控容易过早固化
- 噪声让训练阶段探索更多专家
- 推理时关闭噪声，获得更稳定输出

**Q6：Switch Transformer 和普通 top-k MoE 的关键区别是什么？**

- Switch 用 top-1 路由，结构更简单
- 通信与实现复杂度更低
- 代价是每个 token 只使用一个专家，表达组合能力更弱

**Q7：MoE 的主要工程难点是什么？**

- 路由均衡
- 多设备通信
- expert capacity 控制
- 训练稳定性与吞吐量平衡

**Q8：什么时候不该用 MoE？**

- 模型本身不大，稀疏化收益不明显
- 训练环境不支持高效分布式通信
- 任务规模小，工程复杂度高于收益

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | MoE 核心思想是什么？ | 多专家 + 条件路由，只激活少数参数 |
| 2 | 为什么需要 gate？ | 决定当前输入该交给哪个专家处理 |
| 3 | 为什么要负载均衡？ | 防止 expert collapse，避免参数浪费 |
| 4 | Noisy Gating 干什么？ | 训练时促进探索，推理时关闭噪声保稳定 |
| 5 | Switch 为什么更高效？ | top-1 路由简化了计算和通信 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我从零实现了 MoE 的 gate、expert、稀疏路由和辅助损失，能解释每个组件的作用
- **工程能力**：我又把同一任务压缩成更紧凑的 PyTorch 实现，比较了教学实现和工程实现的差异
- **对比思考**：我不只会写代码，还能说明为什么训练阶段要 noisy gating、为什么要做 load balancing，以及什么情况下不适合用 MoE

### 延伸阅读与对比

| 对比维度 | Dense MLP | Switch Transformer | Mixtral 风格 MoE |
|---|---|---|---|
| 激活参数 | 全部 | 每个 token 1 个专家 | 每个 token 2 个专家 |
| 计算复杂度 | 高 | 更低 | 介于两者之间 |
| 路由复杂度 | 无 | 低 | 中 |
| 表达能力 | 稠密统一 | 简单稀疏 | 更强稀疏组合 |

### 进阶探索方向

- 将本 Notebook 的 top-1 改成 top-2，模拟 Mixtral 风格路由
- 在 Transformer FFN 中嵌入 MoE 层，而不是只做二维分类
- 实现 expert capacity 与 token dropping，进一步贴近 Switch Transformer
- 在多 GPU 环境中研究 all-to-all 通信成本
